In [1]:
%pip install astropy scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import subprocess
import gzip
import shutil
import numpy as np
import pandas as pd
from astropy.io import fits

os.makedirs("../data/raw/lss", exist_ok=True)
os.makedirs("processed", exist_ok=True)
os.makedirs("results", exist_ok=True)

base_url = "https://data.sdss.org/sas/dr12/boss/lss/"

files = [
    ("galaxy_DR12v5_CMASS_North.fits.gz", 138873951),
    ("random0_DR12v5_CMASS_North.fits.gz", 3239878693),
]

In [3]:
for name, expected_size in files:
    target = "../data/raw/lss/" + name

    if os.path.isfile(target):
        assert os.path.getsize(target) == expected_size
        continue

    partial = target + ".part"

    subprocess.run(
        [
            "curl", "--fail", "--location",
            "--retry", "3", "--continue-at", "-",
            "--output", partial, base_url + name,
        ],
        check=True,
    )

    assert os.path.getsize(partial) == expected_size
    os.replace(partial, target)

pd.DataFrame({
    "file": [name for name, size in files],
    "GB": [size / 1e9 for name, size in files],
})

** Resuming transfer from byte position 245116928
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2856M  100 2856M    0     0  13.7M      0  0:03:27  0:03:27 --:--:-- 14.1M


,file,GB
0,galaxy_DR12v5_CMASS_North.fits.gz,0.138874
1,random0_DR12v5_CMASS_North.fits.gz,3.239879


In [4]:
for name, size in files:
    source = "../data/raw/lss/" + name
    target = source[:-3]

    if os.path.isfile(target):
        continue

    with gzip.open(source, "rb") as compressed:
        with open(target + ".part", "wb") as output:
            shutil.copyfileobj(compressed, output, length=8 * 1024 * 1024)

    os.replace(target + ".part", target)

In [5]:
Z_MIN = 0.43
Z_MAX = 0.70
RANDOM_KEEP = 0.10
CHUNK_SIZE = 200000

np.random.seed(42)
records = []

for kind, name in [
    ("galaxy", files[0][0]),
    ("random", files[1][0]),
]:
    source = "../data/raw/lss/" + name[:-3]
    target = "processed/boss_cmass_north_" + kind + ".csv.gz"

    columns = ["RA", "DEC", "Z", "WEIGHT_FKP"]

    if kind == "galaxy":
        columns += ["WEIGHT_SYSTOT", "WEIGHT_CP", "WEIGHT_NOZ"]

    selected_count = 0

    with fits.open(source, memmap=True) as hdul:
        table = hdul[1].data
        total_count = len(table)

        with gzip.open(target + ".part", "wt") as output:
            for first in range(0, total_count, CHUNK_SIZE):
                last = min(first + CHUNK_SIZE, total_count)
                block = table[first:last]

                df = pd.DataFrame({
                    col.lower(): np.array(block[col], dtype=np.float64)
                    for col in columns
                })
                df["row"] = np.arange(first, last)

                selected = (df["z"] > Z_MIN) & (df["z"] < Z_MAX)

                if kind == "random":
                    selected &= np.random.random(len(df)) < RANDOM_KEEP

                df = df.loc[selected].copy()

                assert np.isfinite(df.values).all()

                df["weight"] = df["weight_fkp"]

                if kind == "galaxy":
                    df["weight"] *= (
                        df["weight_systot"]
                        * (df["weight_cp"] + df["weight_noz"] - 1)
                    )

                assert (df["weight"] > 0).all()

                df.to_csv(output, index=False, header=(first == 0))
                selected_count += len(df)

    os.replace(target + ".part", target)

    records.append({
        "catalog": kind,
        "raw_rows": total_count,
        "selected_rows": selected_count,
        "z_min": Z_MIN,
        "z_max": Z_MAX,
        "keep_probability": 1.0 if kind == "galaxy" else RANDOM_KEEP,
    })

selection = pd.DataFrame(records)
selection.to_csv("results/boss_selection.csv", index=False)

selection

,catalog,raw_rows,selected_rows,z_min,z_max,keep_probability
0,galaxy,618806,568776,0.43,0.7,1.0
1,random,32151741,2959170,0.43,0.7,0.1
